# Task 2 v4 - Task 1 Candidate Industries + Task 2 Cross-Encoder Reranker

Goal:

1. Use a Task 1 industry model to generate top parent-industry candidates.
2. Expand those parent industries into possible 10-digit subindustries using the taxonomy.
3. Use the trained Task 2 cross-encoder to rank those candidate subindustries.

Important design rule:

- Task 1 may use `LongProfile` to find likely parent industries.
- Task 2 ranking uses only `SegmentName`, `SegmentDescription`, and candidate taxonomy text.

In [7]:
from pathlib import Path
import gc
import html
import json
import math
import os
import random
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
from scipy.special import log_softmax, softmax
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)

## Config

In [8]:
SEED = 42

# ── Repo root (must be first — everything else derives from it) ──────────────
REPO_ROOT = Path("../..").resolve()

# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR          = REPO_ROOT / "data" / "raw"
TASK2_PATH        = DATA_DIR / "task2_subindustry_classification_final.csv"
TASK1_SOURCE_PATH = DATA_DIR / "task1_gecs_classification_final.csv"
SPLIT_PATH        = DATA_DIR / "task2_candidate_reranker_split_assignments.csv"

# ── Weight paths ──────────────────────────────────────────────────────────────
TAXONOMY_PATH     = REPO_ROOT / "weights" / "task2" / "taxonomy_table.csv"
CROSS_ENCODER_DIR = REPO_ROOT / "weights" / "task2" / "cross_encoder"
TASK1_V10_LE      = REPO_ROOT / "weights" / "task1" / "label_encoder.pkl"
TASK1_V10_SCALER  = REPO_ROOT / "weights" / "task1" / "numeric_scaler.pkl"
TASK1_V10_STATE   = REPO_ROOT / "weights" / "task1" / "best_model_state.pt"

OUTPUT_DIR = Path("checkpoints/task2_v10_eval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK2_CROSS_ENCODER_MODEL = "microsoft/deberta-v3-small"
TASK1_FLANGBERT_MODEL     = "SALT-NLP/FLANG-BERT"
LOCAL_FILES_ONLY          = True

# ── Candidate generation ──────────────────────────────────────────────────────
PARENT_TOPK              = 10
MIN_CANDIDATE_LEAFS      = 0
LOW_CONFIDENCE_THRESHOLD = 0.35
FALLBACK_PARENT_TOPK     = 15

# ── Eval ──────────────────────────────────────────────────────────────────────
EVAL_ROW_LIMIT  = None
MAX_LENGTH      = 256
EVAL_BATCH_SIZE = 16

ALPHA_GRID = [0.0, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS    = 1e-12

import sys
sys.path.insert(0, str(REPO_ROOT))

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device    :", DEVICE)
print("repo_root :", REPO_ROOT)
print("output    :", OUTPUT_DIR.resolve())
for p in [TASK2_PATH, TASK1_SOURCE_PATH, TAXONOMY_PATH, SPLIT_PATH,
          CROSS_ENCODER_DIR / "best_state.pt", TASK1_V10_LE, TASK1_V10_STATE]:
    print(f"  {'OK' if p.exists() else 'MISSING'} {p}")


device    : cpu
repo_root : /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone
output    : /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/notebooks/t2/checkpoints/task2_v10_eval
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/data/raw/task2_subindustry_classification_final.csv
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/data/raw/task1_gecs_classification_final.csv
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/weights/task2/taxonomy_table.csv
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/data/raw/task2_candidate_reranker_split_assignments.csv
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/weights/task2/cross_encoder/best_state.pt
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/weights/task1/label_encoder.pkl
  OK /Users/meetpatel/Documents/capstone/depaul-morningstar-capstone/weights/task1/best_model_state.pt


## Text and Metric Helpers

In [9]:
SMART_TRANS = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-", "\u2212": "-", "\xa0": " ", "\ufeff": ""
})
WS_RE = re.compile(r"\s+")
URL_RE = re.compile(r"http\S+|www\.\S+")
NUMBER_RE = re.compile(r"\b\d[\d,\.]*\b")
NON_ALPHA_RE = re.compile(r"[^a-z\s]")


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value)).translate(SMART_TRANS)
    return WS_RE.sub(" ", text).strip()


def clean_code(value, width=None):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    digits = re.sub(r"\D+", "", text)
    if width and digits:
        return digits.zfill(width)
    return digits


def light_clean(value):
    text = normalize_text(value).lower()
    text = URL_RE.sub(" ", text)
    return WS_RE.sub(" ", text).strip()


DOMAIN_STOPWORDS = {
    "company", "companies", "firm", "firms", "corporation", "corp", "inc", "ltd", "plc", "sa",
    "segment", "segments", "business", "businesses", "operation", "operations", "operating",
    "service", "services", "product", "products", "provide", "provides", "providing", "provided",
    "offer", "offers", "offering", "offered", "include", "includes", "including", "included",
    "primarily", "geographically", "revenue", "revenues", "generate", "generates", "generated",
    "generating", "derive", "derives", "derived", "deriving", "also", "well", "various",
    "mainly", "majority", "largest", "main", "across", "engage", "engaged", "engages",
    "engaging", "consist", "consists", "consisting", "comprise", "comprises", "manage",
    "manages", "managed", "managing", "management", "produce", "produces", "produced",
    "producing", "production", "develop", "develops", "developed", "developing", "development",
    "manufacture", "manufactures", "manufactured", "manufacturing", "manufacturer", "distribute",
    "distributes", "distributed", "distributing", "distribution", "distributor", "market",
    "markets", "marketed", "marketing", "sell", "sells", "sold", "selling", "sale", "sales",
    "use", "uses", "used", "using", "addition", "additional", "additionally", "global",
    "globally", "worldwide", "international", "domestic", "year", "years", "annual", "quarterly",
    "report", "reports", "reportable", "reported", "reporting", "customer", "customers",
    "client", "clients", "region", "regions", "regional", "area", "areas", "country",
    "countries", "north", "south", "east", "west", "american", "america", "europe",
    "european", "asia", "asian", "africa", "african", "pacific", "china", "chinese",
    "japan", "japanese", "india", "indian", "united", "states", "world", "based",
    "primary", "key", "large", "small", "new", "high", "low", "approximately", "per",
    "cent", "percent", "million", "billion", "thousand",
}
BOILERPLATE_PHRASES = [
    "the company is", "the company has", "the company also", "geographically the company",
    "the company generates", "the company derives", "the company provides", "the company operates",
    "the company offers", "the company manufactures", "the company s main",
    "the company s reportable", "the company s operating segment", "the company s operating segments",
    "the firm is", "the firm has", "as well as", "rest of the world", "rest of world",
    "north america", "south america", "asia pacific", "middle east", "latin america",
]


def heavy_clean(value):
    text = light_clean(value)
    text = NUMBER_RE.sub(" ", text)
    text = NON_ALPHA_RE.sub(" ", text)
    for phrase in BOILERPLATE_PHRASES:
        text = text.replace(phrase, " ")
    tokens = [t for t in text.split() if len(t) > 1 and t not in DOMAIN_STOPWORDS]
    return WS_RE.sub(" ", " ".join(tokens)).strip()


def first_words(value, n=140):
    return " ".join(str(value).split()[:n])


def ranking_metrics(true_leafs, ranked_leafs, ks=(1, 3, 5)):
    true_leafs = list(map(str, true_leafs))
    metrics = {}
    for k in ks:
        metrics[f"top{k}_accuracy"] = float(np.mean([
            true in list(map(str, preds[:k]))
            for true, preds in zip(true_leafs, ranked_leafs)
        ]))
    reciprocal_ranks = []
    for true, preds in zip(true_leafs, ranked_leafs):
        preds = list(map(str, preds))
        reciprocal_ranks.append(1.0 / (preds.index(true) + 1) if true in preds else 0.0)
    metrics["mrr"] = float(np.mean(reciprocal_ranks))
    return metrics


def classification_metrics(y_true, y_pred, prefix="test"):
    return {
        f"{prefix}_accuracy": float(accuracy_score(y_true, y_pred)),
        f"{prefix}_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        f"{prefix}_weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }


def top10_leaf_macro_f1(y_true, y_pred):
    top10 = pd.Series(y_true).value_counts().head(10).index.astype(str).tolist()
    mask = pd.Series(y_true).isin(top10).to_numpy()
    return float(f1_score(np.asarray(y_true)[mask], np.asarray(y_pred)[mask], labels=top10, average="macro", zero_division=0))


def print_metrics(title, metrics):
    print(f"\n{title}")
    for key, value in metrics.items():
        if isinstance(value, (float, np.floating, int, np.integer)):
            print(f"{key:28s} {float(value):.4f}")
        else:
            print(f"{key:28s} {value}")

## Load Task 2, Merge LongProfile for Task 1, and Reuse Existing Split

In [10]:
for path in [TASK2_PATH, TASK1_SOURCE_PATH, TAXONOMY_PATH, SPLIT_PATH,
             CROSS_ENCODER_DIR / "best_state.pt", TASK1_V10_LE, TASK1_V10_STATE]:
    assert path.exists(), f"Missing: {path}"

task2 = pd.read_csv(TASK2_PATH).reset_index().rename(columns={"index": "row_index"})
task2["leaf"]   = task2["SubIndustry"].map(lambda x: clean_code(x, width=10))
task2["parent"] = task2["leaf"].str[:8]
task2["SegmentName"]        = task2["SegmentName"].fillna("").map(normalize_text)
task2["SegmentDescription"] = task2["SegmentDescription"].fillna("").map(normalize_text)

task1_source = pd.read_csv(
    TASK1_SOURCE_PATH,
    usecols=["CompanyId", "AsOfDate", "LongProfile", "MstarGlobal"],
    dtype={"CompanyId": str, "AsOfDate": str},
)
task1_source["LongProfile"] = task1_source["LongProfile"].fillna("").map(normalize_text)

exact_profile = (
    task1_source.sort_values(["CompanyId", "AsOfDate"])
    .drop_duplicates(["CompanyId", "AsOfDate"])[["CompanyId", "AsOfDate", "LongProfile"]]
)
company_profile = (
    task1_source[task1_source["LongProfile"].ne("")]
    .drop_duplicates(["CompanyId"])[["CompanyId", "LongProfile"]]
    .rename(columns={"LongProfile": "LongProfile_company"})
)

task2 = task2.merge(exact_profile,   on=["CompanyId", "AsOfDate"], how="left")
task2 = task2.merge(company_profile, on="CompanyId",               how="left")
task2["LongProfile"] = task2["LongProfile"].fillna(task2["LongProfile_company"]).fillna("")
task2 = task2.drop(columns=["LongProfile_company"])

splits = pd.read_csv(SPLIT_PATH, dtype={"leaf": str, "parent": str})
assert len(splits) == len(task2), "Split length mismatch."
assert (splits["leaf"].astype(str).to_numpy() == task2["leaf"].astype(str).to_numpy()).all(), "Split leaf order mismatch."
task2["split"] = splits["split"].to_numpy()

task2["segment_text"] = (
    "[SEGMENT_NAME] " + task2["SegmentName"].fillna("")
    + " [SEGMENT_DESCRIPTION] " + task2["SegmentDescription"].fillna("")
)

if EVAL_ROW_LIMIT is not None:
    val_df  = task2[task2["split"].eq("val")].sample(n=min(EVAL_ROW_LIMIT,  task2["split"].eq("val").sum()),  random_state=SEED).reset_index(drop=True)
    test_df = task2[task2["split"].eq("test")].sample(n=min(EVAL_ROW_LIMIT, task2["split"].eq("test").sum()), random_state=SEED).reset_index(drop=True)
else:
    val_df  = task2[task2["split"].eq("val")].reset_index(drop=True)
    test_df = task2[task2["split"].eq("test")].reset_index(drop=True)

print(f"Task 2 rows : {len(task2):,}")
print(task2["split"].value_counts().to_string())
print(f"LongProfile : {task2['LongProfile'].ne('').mean():.4f}")
print(f"val={len(val_df):,}  test={len(test_df):,}")


Task 2 rows : 27,537
split
train    16691
test      5425
val       5421
LongProfile : 1.0000
val=5,421  test=5,425


## Load Taxonomy and Candidate Maps

In [11]:
taxonomy = pd.read_csv(TAXONOMY_PATH, dtype=str).fillna("")
taxonomy["industry_code"] = taxonomy["industry_code"].map(lambda x: clean_code(x, width=8))
taxonomy["subindustry_code"] = taxonomy["subindustry_code"].map(lambda x: clean_code(x, width=10))

taxonomy_text_by_leaf = taxonomy.set_index("subindustry_code")["taxonomy_text"].to_dict()
leaf_to_parent = taxonomy.set_index("subindustry_code")["industry_code"].to_dict()
leaf_name = taxonomy.set_index("subindustry_code")["subindustry_name"].to_dict()
parent_name = taxonomy.drop_duplicates("industry_code").set_index("industry_code")["industry_name"].to_dict()
leaves_by_parent = taxonomy.groupby("industry_code")["subindustry_code"].apply(lambda x: list(dict.fromkeys(x))).to_dict()
all_parent_codes = sorted(leaves_by_parent)

missing_leafs = sorted(set(task2["leaf"].unique()) - set(taxonomy_text_by_leaf))
missing_parents = sorted(set(task2["parent"].unique()) - set(leaves_by_parent))
print("taxonomy leaf classes:", len(taxonomy_text_by_leaf))
print("taxonomy parent classes:", len(leaves_by_parent))
print("missing leaf labels from taxonomy:", len(missing_leafs), missing_leafs[:10])
print("missing parent labels from taxonomy:", len(missing_parents), missing_parents[:10])

taxonomy leaf classes: 445
taxonomy parent classes: 145
missing leaf labels from taxonomy: 11 ['1013002011', '1028006009', '1035002002', '1041001002', '1042001001', '2072002005', '2072002006', '2072002007', '2072003003', '3102004003']
missing parent labels from taxonomy: 0 []


## Task 1 Parent Candidate Generator

In [12]:
def align_cols(mat, src_classes, target_classes):
    src_classes    = np.asarray(src_classes).astype(str)
    target_classes = np.asarray(target_classes).astype(str)
    if list(src_classes) == list(target_classes):
        return mat
    pos = {c: i for i, c in enumerate(src_classes)}
    return mat[:, [pos[c] for c in target_classes]]


### Optional v8 Segment-Focused SEC-BERT Stream

In [ ]:
# ── FLANG-BERT v10 inference on Task 2 eval splits ──────────────────────────
import pickle
from serving_app.inference import (
    Task1FlangBertRanker, ModelPaths,
    build_task1_text, compute_numeric_features,
)

# Task 2 CSV is missing Task 1 numeric/meta columns.
# Fill with safe defaults so compute_numeric_features doesn't silently miscompute.
_t1_extra_cols = {
    "Revenue"                    : 0.0,
    "total_revenue_company_as_of": 0.0,
    "revenue_share"              : 0.0,
    "is_largest_share_segment"   : 0,
    "segment_desc_imputed"       : False,
    "MstarGlobal"                : "",   # → sector/group tokens stay "[]" "[]" (correct dropout path)
}
for col, default in _t1_extra_cols.items():
    if col not in val_df.columns:
        val_df[col]  = default
        test_df[col] = default

print("Columns defaulted (missing in Task 2 CSV):",
      [c for c in _t1_extra_cols if c not in task2.columns])

# ── Run inference ─────────────────────────────────────────────────────────────
_paths  = ModelPaths.discover(REPO_ROOT)
_ranker = Task1FlangBertRanker(_paths, DEVICE, LOCAL_FILES_ONLY)

print("Running FLANG-BERT v10 inference on val split ...")
val_v10_logits  = _ranker.log_scores(val_df.reset_index(drop=True))
print(f"  val  logits: {val_v10_logits.shape}")

print("Running FLANG-BERT v10 inference on test split ...")
test_v10_logits = _ranker.log_scores(test_df.reset_index(drop=True))
print(f"  test logits: {test_v10_logits.shape}")

np.save(OUTPUT_DIR / "v10_val_logits.npy",  val_v10_logits)
np.save(OUTPUT_DIR / "v10_test_logits.npy", test_v10_logits)
print("Logits saved to", OUTPUT_DIR)


Columns defaulted (missing in Task 2 CSV): ['Revenue', 'total_revenue_company_as_of', 'revenue_share', 'is_largest_share_segment', 'segment_desc_imputed', 'MstarGlobal']
Running FLANG-BERT v10 inference on val split ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: SALT-NLP/FLANG-BERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Build and Save Task 1 Parent Candidates

In [ ]:
import pickle

with open(TASK1_V10_LE, "rb") as f:
    le_v10 = pickle.load(f)
raw_classes  = np.asarray(le_v10.classes_).astype(str)
task1_classes = np.array([clean_code(c, width=8) for c in raw_classes])

assert set(task1_classes) >= set(task2["parent"].unique()),     "FLANG-BERT v10 label encoder is missing Task 2 parent codes."
print(f"task1_classes: {len(task1_classes)} industries — OK")


def blended_task1_log_scores(frame, split):
    """Single stream: FLANG-BERT v10 logits (pre-computed above)."""
    logits = val_v10_logits if split == "val" else test_v10_logits
    assert len(logits) == len(frame),         f"Logit rows {len(logits)} != frame rows {len(frame)}"
    return logits, {"flangbert_v10": 1.0}


def parent_candidates_from_scores(frame, log_scores, split_name):
    probs = softmax(log_scores, axis=1)
    order = np.argsort(-probs, axis=1)
    rows  = []
    for i, (_, row) in enumerate(frame.iterrows()):
        top_idx = order[i, :max(PARENT_TOPK, FALLBACK_PARENT_TOPK)]
        codes   = task1_classes[top_idx].astype(str).tolist()
        scores  = probs[i, top_idx].astype(float).tolist()
        rows.append({
            "source_row_id"    : i,
            "row_index"        : int(row["row_index"]),
            "CompanyId"        : row["CompanyId"],
            "true_parent"      : row["parent"],
            "true_leaf"        : row["leaf"],
            "top_parent_codes" : json.dumps(codes),
            "top_parent_scores": json.dumps(scores),
            "top1_parent"      : codes[0],
            "top1_parent_score": scores[0],
        })
    cand = pd.DataFrame(rows)
    cand.to_csv(OUTPUT_DIR / f"task1_best_parent_candidates_{split_name}.csv", index=False)
    return cand


def parent_topk_recall(cand, ks=(1, 3, 5, 10)):
    parsed = cand["top_parent_codes"].map(json.loads)
    truth  = cand["true_parent"].astype(str).tolist()
    return {f"parent_top{k}_recall": float(np.mean([t in codes[:k] for t, codes in zip(truth, parsed)])) for k in ks}


start = time.time()
val_task1_log,  task1_weights = blended_task1_log_scores(val_df,  "val")
test_task1_log, _             = blended_task1_log_scores(test_df, "test")
val_parent_candidates  = parent_candidates_from_scores(val_df,  val_task1_log,  "val")
test_parent_candidates = parent_candidates_from_scores(test_df, test_task1_log, "test")

parent_metrics = {
    "val"           : parent_topk_recall(val_parent_candidates),
    "test"          : parent_topk_recall(test_parent_candidates),
    "stream_weights": task1_weights,
    "seconds"       : round(time.time() - start, 1),
}
with open(OUTPUT_DIR / "task1_parent_candidate_metrics.json", "w") as f:
    json.dump(parent_metrics, f, indent=2)
print(json.dumps(parent_metrics, indent=2))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19324.11it/s]
[transformers] BertModel LOAD REPORT from: nlpaueb/sec-bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task 1 candidate stream weights: {'segment_svc': np.float64(0.26932045461705467), 'profile_svc': np.float64(0.5323578799173604), 'v8_segfocus_secbert': np.float64(0.19832166546558497)}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18236.90it/s]
[transformers] BertModel LOAD REPORT from: nlpaueb/sec-bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task 1 candidate stream weights: {'segment_svc': np.float64(0.26932045461705467), 'profile_svc': np.float64(0.5323578799173604), 'v8_segfocus_secbert': np.float64(0.19832166546558497)}
{
  "val": {
    "parent_top1_recall": 0.9000184467810367,
    "parent_top3_recall": 0.9719608928242022,
    "parent_top5_recall": 0.9843202361187973,
    "parent_top10_recall": 0.9940970300682531
  },
  "test": {
    "parent_top1_recall": 0.9076497695852535,
    "parent_top3_recall": 0.9725345622119815,
    "parent_top5_recall": 0.9861751152073732,
    "parent_top10_recall": 0.9937327188940093
  },
  "stream_weights": {
    "segment_svc": 0.26932045461705467,
    "profile_svc": 0.5323578799173604,
    "v8_segfocus_secbert": 0.19832166546558497
  },
  "seconds": 182.2
}


## Load Task 2 Cross-Encoder

In [ ]:
def build_pair_text(segment_text, taxonomy_text):
    return (
        f"[SEGMENT] {normalize_text(segment_text)} "
        f"[CANDIDATE_TAXONOMY] {normalize_text(taxonomy_text)} "
        "[QUESTION] Does this segment belong to this subindustry?"
    ).strip()


class PairDataset(Dataset):
    def __init__(self, pair_df, tokenizer, max_length=MAX_LENGTH):
        self.texts = pair_df["pair_text"].astype(str).tolist()
        self.labels = pair_df["label"].astype(np.float32).to_numpy()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


def masked_mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom


class SegmentTaxonomyCrossEncoder(nn.Module):
    def __init__(self, model_name=TASK2_CROSS_ENCODER_MODEL, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, local_files_only=LOCAL_FILES_ONLY)
        self.encoder.float()
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        if getattr(outputs, "pooler_output", None) is not None:
            pooled = outputs.pooler_output
        else:
            pooled = masked_mean_pool(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        pooled = pooled.to(self.classifier.weight.dtype)
        return self.classifier(pooled).squeeze(-1)


@torch.no_grad()
def predict_pair_logits(model, pair_df, tokenizer, batch_size=EVAL_BATCH_SIZE):
    ds = PairDataset(pair_df, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    logits = []
    model.eval()
    for batch in tqdm(loader, desc="Task 2 cross-encoder scoring", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        logits.append(out.float().cpu().numpy())
    return np.concatenate(logits)

## Build Candidate Subindustries and Score Them

In [ ]:
def candidate_leafs_from_parent_codes(parent_codes, min_candidates=MIN_CANDIDATE_LEAFS):
    seen = []
    for parent in parent_codes:
        for leaf in leaves_by_parent.get(str(parent), []):
            if leaf not in seen:
                seen.append(leaf)
    if len(seen) < min_candidates:
        frequent_leafs = task2["leaf"].value_counts().index.astype(str).tolist()
        for leaf in frequent_leafs:
            if leaf not in seen:
                seen.append(leaf)
            if len(seen) >= min_candidates:
                break
    return seen


def candidate_rows_for_frame(frame, parent_candidates, split_name):
    parent_by_pos = {int(row.source_row_id): row for row in parent_candidates.itertuples(index=False)}
    rows = []
    for pos, row in frame.reset_index(drop=True).iterrows():
        cand_row = parent_by_pos[pos]
        parent_codes = json.loads(cand_row.top_parent_codes)
        parent_scores = json.loads(cand_row.top_parent_scores)
        use_k = PARENT_TOPK
        if parent_scores and parent_scores[0] < LOW_CONFIDENCE_THRESHOLD:
            use_k = FALLBACK_PARENT_TOPK
        parent_codes = parent_codes[:use_k]
        parent_scores = parent_scores[:use_k]
        score_by_parent = {p: s for p, s in zip(parent_codes, parent_scores)}
        candidate_leafs = candidate_leafs_from_parent_codes(parent_codes)
        for leaf in candidate_leafs:
            parent = leaf[:8]
            tax_text = taxonomy_text_by_leaf.get(leaf, f"[SUBINDUSTRY_CODE] {leaf} [INDUSTRY_CODE] {parent}")
            rows.append({
                "source_row_id": pos,
                "row_index": int(row["row_index"]),
                "split": split_name,
                "true_leaf": row["leaf"],
                "true_parent": row["parent"],
                "candidate_leaf": leaf,
                "candidate_parent": parent,
                "candidate_parent_name": parent_name.get(parent, ""),
                "candidate_leaf_name": leaf_name.get(leaf, ""),
                "parent_score": float(score_by_parent.get(parent, EPS)),
                "pair_text": build_pair_text(row["segment_text"], tax_text),
                "label": int(leaf == row["leaf"]),
            })
    return pd.DataFrame(rows)


def score_split(frame, parent_candidates, split_name):
    pair_df = candidate_rows_for_frame(frame, parent_candidates, split_name)
    tokenizer = AutoTokenizer.from_pretrained(TASK2_CROSS_ENCODER_MODEL, local_files_only=LOCAL_FILES_ONLY)
    model = SegmentTaxonomyCrossEncoder().to(DEVICE)
    model.load_state_dict(torch.load(CROSS_ENCODER_DIR / "best_state.pt", map_location=DEVICE))
    pair_df["cross_encoder_logit"] = predict_pair_logits(model, pair_df.assign(label=0), tokenizer)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pair_df


start = time.time()
val_scored = score_split(val_df, val_parent_candidates, "val")
test_scored = score_split(test_df, test_parent_candidates, "test")
val_scored.to_csv(OUTPUT_DIR / "scored_pairs_val_task1_best_topk.csv", index=False)
test_scored.to_csv(OUTPUT_DIR / "scored_pairs_test_task1_best_topk.csv", index=False)
print(f"Scoring seconds: {time.time() - start:.1f}")
print("avg val candidates:", round(val_scored.groupby("source_row_id").size().mean(), 2))
print("avg test candidates:", round(test_scored.groupby("source_row_id").size().mean(), 2))
print("val candidate coverage:", val_scored.groupby("source_row_id")["label"].max().mean())
print("test candidate coverage:", test_scored.groupby("source_row_id")["label"].max().mean())

[transformers] The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 10759.22it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.

Scoring seconds: 6108.8
avg val candidates: 50.3
avg test candidates: 50.44
val candidate coverage: 0.9631064379265818
test candidate coverage: 0.9633179723502304


## Tune Parent Prior Alpha on Validation and Evaluate Test

In [ ]:
def rank_scored_pairs(scored_pairs, alpha):
    df = scored_pairs.copy()
    df["final_score"] = df["cross_encoder_logit"] + alpha * np.log(df["parent_score"].clip(EPS, 1.0))
    ranked = []
    truth = []
    best_rows = []
    for row_id, group in df.groupby("source_row_id", sort=True):
        group = group.sort_values("final_score", ascending=False)
        ranked.append(group["candidate_leaf"].astype(str).tolist())
        truth.append(str(group["true_leaf"].iloc[0]))
        best_rows.append(group.iloc[0].to_dict())
    pred = [r[0] if r else "" for r in ranked]
    return truth, ranked, np.array(pred, dtype=str), pd.DataFrame(best_rows)


def evaluate_scored_pairs(scored_pairs, alpha, prefix):
    truth, ranked, pred, best_rows = rank_scored_pairs(scored_pairs, alpha)
    metrics = ranking_metrics(truth, ranked, ks=(1, 3, 5))
    metrics.update(classification_metrics(np.array(truth), pred, prefix=prefix))
    metrics[f"{prefix}_top10_leaf_macro_f1"] = top10_leaf_macro_f1(np.array(truth), pred)
    metrics["avg_candidates"] = float(scored_pairs.groupby("source_row_id").size().mean())
    metrics["candidate_coverage"] = float(scored_pairs.groupby("source_row_id")["label"].max().mean())
    return metrics, best_rows


rows = []
for alpha in ALPHA_GRID:
    metrics, _ = evaluate_scored_pairs(val_scored, alpha, prefix="val")
    rows.append({"alpha": alpha, **metrics})
alpha_sweep = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
alpha_sweep.to_csv(OUTPUT_DIR / "alpha_sweep.csv", index=False)
display(alpha_sweep)

best_alpha = float(alpha_sweep.iloc[0]["alpha"])
val_metrics, val_best = evaluate_scored_pairs(val_scored, best_alpha, prefix="val")
test_metrics, test_best = evaluate_scored_pairs(test_scored, best_alpha, prefix="test")
val_best.to_csv(OUTPUT_DIR / "best_predictions_val.csv", index=False)
test_best.to_csv(OUTPUT_DIR / "best_predictions_test.csv", index=False)

final_rows = [
    {"split": "val", "alpha": best_alpha, **val_metrics},
    {"split": "test", "alpha": best_alpha, **test_metrics},
]
final_metrics = pd.DataFrame(final_rows)
final_metrics.to_csv(OUTPUT_DIR / "task1_best_pipeline_metrics.csv", index=False)

with open(OUTPUT_DIR / "task1_best_pipeline_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_alpha": best_alpha,
        "val": val_metrics,
        "test": test_metrics,
        "parent_candidate_metrics": parent_metrics,
    }, f, indent=2)

print_metrics("VAL best", {"alpha": best_alpha, **val_metrics})
print_metrics("TEST best", {"alpha": best_alpha, **test_metrics})
display(final_metrics)

,alpha,top1_accuracy,top3_accuracy,top5_accuracy,mrr,val_accuracy,val_macro_f1,val_weighted_f1,val_top10_leaf_macro_f1,avg_candidates,candidate_coverage
9,2.00,0.595462,0.842096,0.919941,0.724949,0.595462,0.450260,0.567244,0.860981,50.295148,0.963106
8,1.50,0.582918,0.828629,0.914407,0.714416,0.582918,0.445253,0.556004,0.842965,50.295148,0.963106
7,1.25,0.573326,0.818299,0.908135,0.706222,0.573326,0.439933,0.546819,0.829924,50.295148,0.963106
6,1.00,0.559122,0.809814,0.898174,0.695379,0.559122,0.431058,0.532504,0.816510,50.295148,0.963106
5,0.75,0.537724,0.792289,0.884154,0.679356,0.537724,0.416882,0.512479,0.789832,50.295148,0.963106
4,0.50,0.513005,0.770707,0.865707,0.658900,0.513005,0.395698,0.488494,0.764910,50.295148,0.963106
3,0.35,0.489209,0.754473,0.850950,0.641495,0.489209,0.379748,0.465183,0.732767,50.295148,0.963106
2,0.20,0.466519,0.740823,0.834717,0.623562,0.466519,0.357861,0.443520,0.709197,50.295148,0.963106
1,0.10,0.450470,0.728279,0.823280,0.610654,0.450470,0.345405,0.427521,0.692583,50.295148,0.963106
0,0.00,0.432393,0.711861,0.809629,0.595676,0.432393,0.327471,0.409100,0.672023,50.295148,0.963106



VAL best
alpha                        2.0000
top1_accuracy                0.5955
top3_accuracy                0.8421
top5_accuracy                0.9199
mrr                          0.7249
val_accuracy                 0.5955
val_macro_f1                 0.4503
val_weighted_f1              0.5672
val_top10_leaf_macro_f1      0.8610
avg_candidates               50.2951
candidate_coverage           0.9631

TEST best
alpha                        2.0000
top1_accuracy                0.6039
top3_accuracy                0.8512
top5_accuracy                0.9248
mrr                          0.7327
test_accuracy                0.6039
test_macro_f1                0.4553
test_weighted_f1             0.5761
test_top10_leaf_macro_f1     0.8491
avg_candidates               50.4394
candidate_coverage           0.9633


,split,alpha,top1_accuracy,top3_accuracy,top5_accuracy,mrr,val_accuracy,val_macro_f1,val_weighted_f1,val_top10_leaf_macro_f1,avg_candidates,candidate_coverage,test_accuracy,test_macro_f1,test_weighted_f1,test_top10_leaf_macro_f1
0,val,2.0,0.595462,0.842096,0.919941,0.724949,0.595462,0.45026,0.567244,0.860981,50.295148,0.963106,NaN,NaN,NaN,NaN
1,test,2.0,0.603871,0.851244,0.924793,0.732726,NaN,NaN,NaN,NaN,50.439447,0.963318,0.603871,0.45528,0.576133,0.84907


## Output Files

After a full run, `checkpoints/task2_v10_eval/` contains:

- `v10_val_logits.npy` / `v10_test_logits.npy` — FLANG-BERT v10 parent priors
- `task1_best_parent_candidates_val.csv` / `..._test.csv` — top-K parent candidates per row
- `task1_parent_candidate_metrics.json` — parent recall @ 1/3/5/10
- `scored_pairs_val_task1_best_topk.csv` / `..._test.csv` — cross-encoder scored pairs
- `alpha_sweep.csv` — val F1 by alpha
- `task1_best_pipeline_metrics.json` — final val + test metrics
- `best_predictions_val.csv` / `best_predictions_test.csv` — top-1 predictions
